In [1]:
import pandas as pd
from google.cloud import bigquery

client = bigquery.Client(project="poai-research")

def bits_to_difficulty(bits_hex):
    bits = int(bits_hex, 16)
    exponent = bits >> 24
    coefficient = bits & 0xffffff
    target = coefficient * 256 ** (exponent - 3)
    max_target = 0x00ffff * 256 ** (0x1d - 3)  # genesis block's target = "difficulty 1"
    return max_target / target

assert bits_to_difficulty("1d00ffff") == 1.0, "formula is wrong — stop here"

# --- full pull ---
blocks_df = client.query("""
    SELECT * FROM `poai-research.poai_data.blocks_data` ORDER BY number
""").to_dataframe()

volumes_df = client.query("""
    SELECT * FROM `poai-research.poai_data.block_volumes` ORDER BY block_number
""").to_dataframe()

print("blocks:", blocks_df.shape, " volumes:", volumes_df.shape)

# --- merge ---
merged = blocks_df.merge(volumes_df, left_on="number", right_on="block_number", how="inner")
print("merged:", merged.shape)

# --- clean Decimal columns ---
for col in ["total_output_satoshis", "total_output_satoshis_excl_coinbase", "total_fee_satoshis"]:
    merged[col] = merged[col].astype(float)

# --- difficulty ---
merged["difficulty"] = merged["bits"].apply(bits_to_difficulty)

# --- sanity checks — look at these before saving anything ---
print("row count == 810909?", merged.shape[0] == 810909)
print("tx_count mismatches:", (merged["transaction_count"] != merged["tx_count_check"]).sum())
print(merged[["number", "difficulty"]].describe())

# --- only save once the above looks right ---
merged.to_csv("real_bitcoin_blocks_raw.csv", index=False)
print("saved:", merged.shape)

blocks: (810909, 5)  volumes: (810909, 5)
merged: (810909, 10)
row count == 810909? True
tx_count mismatches: 0
              number    difficulty
count       810909.0  8.109090e+05
mean        405454.0  7.821704e+12
std    234089.409057  1.309043e+13
min              0.0  1.000000e+00
25%         202727.0  2.968775e+06
50%         405454.0  1.668515e+11
75%         608181.0  1.300809e+13
max         810908.0  5.732151e+13
saved: (810909, 11)
